## In this notebook

- **Background:** in the Dense assignment you ran a grid of widths and depths on ship images; the overview deck lists network size among the small decisions
- **This notebook:** one decision in detail, the number of units per layer: what we measure, what "dead" means, a test of the Lottery Ticket idea, and what this says about the ships networks
- **By the end you can**
    - say when a too-narrow network fails, and why
    - recognize a dead layer, and what does and does not rescue it
    - say what the Lottery Ticket test on a toy did and did not show

# How big should my NN be ?

Another "small" decision: the size of the network (units per layer). Terms such as ReLU, tanh, He and Glorot initialization are as in the overview deck.

A common story, the **paradox**

- start off training an overly large NN (many units)
- many units end up contributing little
- so shrink the network to what it seems to need
- ... and now it can't be trained

and the folk wisdom (which we test): given a fixed number of layers, **a big NN is easier to train than a small one**.

In this notebook

- we **measure** how size affects training on a small running example
- see what **"dead" units** are, and why a too-narrow net fails
- look at one explanation, the **Lottery Ticket** idea, and test it on a toy
- tie back to the **ships networks** you trained
- end with a **practical rule**, and say whether the paradox survives

## Measuring it: units per layer

**Setup:** 1000 points on two concentric rings (two balanced classes, noise $0.1$); 4 hidden layers and one output unit (a logit); Adam (a gradient-descent variant) with learning rate $0.01$; 200 epochs (full passes over the data); cross-entropy loss; **20 random starting points (seeds)** for each size. ReLU uses He initialization, tanh uses Glorot initialization. Each dot is one run; "success" means $90\%$ training accuracy (all numbers here are training accuracy).

<img src="images/netsize_data.png" width=25%>

<img src="images/netsize_seeds.png" width=75%>

- **5 or more units** per layer: nearly every run succeeds (ReLU 20/20 at 5 and 10 units, 19/20 at 50 with one run stalling at $84\%$; tanh 20/20). Width 4 was not tried: the threshold lies between 3 and 5
- **3 units**: ReLU 10/20, tanh 17/20. **2 units**: ReLU 0/20, tanh 1/20
- at 2 to 3 units many ReLU runs sit at **exactly** $50\%$, which is chance for two balanced classes (the net outputs one class for every point: with a whole hidden layer dead, the output is a constant); tanh runs never do
- the other failures stop partway, between about $63\%$ and $89\%$
- so the data show a **threshold** (too narrow fails). They do **not** show that 50 units are easier to train than 5: both succeed

## What "dead" means, and what it costs

A ReLU unit is **dead** when its output is $0$ for every training example.
- its slope is $0$ everywhere it is used, so its incoming weights and bias receive no gradient and it cannot recover (see the overview deck)
- different from a unit with *small weights*: that unit is still active and still learns

<img src="images/netsize_dead.png" width=60%>

- *Setup:* the runs of the previous slide, counted on the training data at the end of training; no error bars
- nets that train well (blue) still have about $10\%$ to $19\%$ of their ReLU units dead (5 or more units per layer): **a normal 10% to 19% is not a warning sign**
- the narrow nets have more: $39\%$ of all units at 2 units, $28\%$ at 3
- but **across widths** the dead fraction does not tell success from failure
    - successful 3-unit nets: $17.5\%$ dead, more than 5-unit nets ($15\%$)
    - within 3 units the failures do have more, because they include dead layers
- a **whole layer** dead does tell them apart (next slides)
- whether the $10\%$ to $19\%$ differ between widths 5, 10 and 50 (they may, by about a factor of 2) was not tested

## A net that is too narrow: capacity or trainability?

Two different reasons a narrow net can fail
- **capacity**: no setting of the weights solves the task
- **trainability**: a good setting exists, but training does not find it

A fixed learning rate and number of epochs cannot tell them apart, so we also tried **5 times more epochs** (1000, learning rate $0.01$) and a **5 times larger learning rate** ($0.05$, 200 epochs). Runs reaching $90\%$, out of 20:

| units | activation | 200 epochs, lr $0.01$ (baseline) | 1000 epochs | lr $0.05$ |
|---|---|---|---|---|
| 2 | ReLU | 0 | 2 | 0 |
| 3 | ReLU | 10 | 10 | 11 |
| 2 | tanh | 1 | 7 | 3 |
| 3 | tanh | 17 | 20 | 17 |

- **tanh**: **more epochs** help most (2 units: 1 to 7; 3 units: 17 to 20)
    - a larger learning rate helps a little at 2 units (1 to 3), not at 3 units
    - no tanh run ends at chance (tanh units saturate, they do not die as ReLU units do)
    - 13 of 20 runs at 2 units still fall short after 1000 epochs; we did not check whether they were still improving, so capacity is not ruled out for those
- **ReLU**: the runs stuck at exactly chance (14 of 20 at 2 units, 7 of 20 at 3) are always runs with a **whole hidden layer dead**
    - the same 14 and 7 stay at chance in all three variants; none of the successful runs has a dead layer
    - more epochs or a larger learning rate do not revive them (the 2 extra successes at 1000 epochs came from runs that were not dead)
- even 2-unit nets can reach high accuracy (best run with 1000 epochs: $97\%$ ReLU, $95\%$ tanh), so **capacity exists**
- the main cause of the narrow failures here is **trainability**, of two kinds: slow training (tanh) and dead layers (ReLU)
    - the 13 tanh runs at 2 units that still fall short are unresolved

## Narrow ReLU nets often start dead (part of the story)

Is a whole layer dead from the start, or does it die during training? We checked the same ReLU nets **before any training** (400 seeds per width):

| units per layer | seeds with a whole hidden layer dead at initialization |
|---|---|
| 2 | $43.5\%$ |
| 3 | $12.0\%$ |
| 5 | $0\%$ |
| 10 | $0\%$ |

What happened to the 20 ReLU runs of the earlier slides (first 20 seeds)

| units | reached 90% | chance, dead at initialization | chance, died early in training | stalled partway |
|---|---|---|---|---|
| 2 | 0 | 9 | 5 | 6 |
| 3 | 10 | 2 | 5 | 3 |
| 5 | 20 | 0 | 0 | 0 |

- why a unit can be dead at the start
    - after the first layer a layer's inputs are all non-negative, and the biases start at $0$
    - so a unit whose weights are all negative on the active inputs never switches on
    - with only 2 or 3 units per layer it is much more likely that **all** of them are like that
    - a dead layer feeds zeros to the next one, so every layer after it is dead too
- so the chance of a dead layer **falls quickly with width**: one mechanism behind "a wider net is safer". It explains the starts that are dead, not the runs that **die early in training** (when and why was not examined)
- what could fix it
    - a different seed (different initial weights) rescues a net that starts dead
    - keeping some slope for negative inputs should too (ReLU variants; not tested here)

## What about the ships networks?

In the Dense assignment you trained ReLU nets with 4 to 256 units per layer on $6400$ inputs, and the **width-4 nets were erratic**. Is that the dead-layer story? We re-ran the assignment's recipe (gray-scale ships, standardized; 30 epochs; Adam, learning rate $0.001$; batch 64; default Glorot initialization; **20 seeds**; a different random split than yours) for widths 4 and 16:

| hidden layers $L$ | width | runs with a whole layer dead (start or end) | validation accuracy, mean (min to max) |
|---|---|---|---|
| 1 | 4 | 0 of 20 | $91.5\%$ ($86.3\%$ to $94.2\%$) |
| 2 | 4 | 0 of 20 | $91.0\%$ ($87.5\%$ to $94.0\%$) |
| 3 | 4 | 0 of 20 | $91.2\%$ ($87.3\%$ to $94.7\%$) |
| 1 | 16 | 0 of 20 | $93.8\%$ ($90.8\%$ to $96.2\%$) |
| 2 | 16 | 0 of 20 | $94.2\%$ ($92.0\%$ to $96.0\%$) |
| 3 | 16 | 0 of 20 | $94.5\%$ ($91.5\%$ to $95.7\%$) |

- **no whole layer was dead in any of the 120 runs**: the dead-layer mechanism of the toy does **not** explain the ships results
- width 4 is still **erratic and worse**: identical nets differ by 6 to 8 points only because of the seed, and average about 3 points below width 16
- why was **not tested**. A guess: a 4-unit bottleneck passes little of the information in $6400$ pixels, and 400 validation examples add noise of about 1 to 1.5 points
- we reported only **whole** dead layers, not partly dead ones, and did not run narrower ships nets
- the toy's threshold (about 5 units) **may not** carry over: here the first layer sees $6400$ signed inputs, not $2$

## The Lottery Ticket idea

[The Lottery Ticket Hypothesis](https://arxiv.org/abs/1803.03635) (Frankle and Carbin, 2018) is about pruning and sparse trainability, and offers one reading of the paradox.

<img src="images/netsize_pruned.png" width=80%>

- a big, randomly initialized NN contains small sub-networks ("winning tickets"), **sparse** (most weights absent) as drawn
- trained in isolation **from the same initial weights**, a winning ticket can match the accuracy of the full network
- found by: train the big NN, **prune** (delete) the smallest-magnitude weights, reset the survivors to their original initial values, retrain
    - a **mask** records which weights survive; the paper repeats the prune-and-reset cycle several times
- the same sparse sub-network trained from a **fresh** random initialization does worse (in the paper's experiments; later work found the picture depends on the learning rate and the network size)
- one reading: a big NN is easier to train because it holds many candidate sub-networks, a bigger lottery (the paper does not state it this way, and we do not test it)

## Testing sparse training on a toy

**Setup:** the two-ring data again; a net with 4 hidden ReLU layers of $50$ units ($7650$ weights; biases are not counted and never pruned, so a unit with no weights left still outputs its bias); Adam, 300 epochs (not 200 as before); **30 seeds**. Train the full net, then keep only the largest-magnitude weights, in **one** pruning step, in two ways: **globally** (one threshold for all layers) or **per layer** (the same fraction in every layer). "Retrain" means training the masked net **again from initial weights**, original or new, not fine-tuning the trained ones. The **random mask** keeps the same number of weights in each layer as the magnitude mask, at random positions.

<img src="images/netsize_lottery.png" width=90%>

- **pruned, not retrained** (pink): about chance: mean accuracy $50\%$ to $55\%$ (not shown in the figure, which counts runs reaching $90\%$), so the pruned net must be trained again
- **global pruning**: $10\%$ kept, 27 or 28 of 30 runs succeed
    - $5\%$ ($382$ weights): original initial weights 15, new 16, random mask 9
    - $2\%$ and $1\%$: (almost) none works
    - why (checked on 5 seeds): at $2\%$ or less the **output layer** typically keeps no weights at all
    - at $5\%$ it keeps 2 output weights and about 2 live units in the last hidden layer, but **60 of the 100 first-layer weights**: the big middle layers are thinned instead
- **per-layer pruning**: $20\%$ kept, all variants succeed 30/30; $10\%$ ($765$ weights): original 20, new 17, random 18; $5\%$ or less: none. At $5\%$ it keeps only 5 of the 100 first-layer weights, which starves the inputs

## What the toy does and does not show

- a sparse net **can** be trained to $90\%$: with $20\%$ of the weights always (per layer), with $10\%$ in $57\%$ to $93\%$ of runs depending on the pruning and the mask, with $5\%$ in about half the runs (global pruning only)
- **how the weights are spread over the layers matters**: at the same sparsity global pruning kept a trainable net at $5\%$ and per-layer pruning did not (it starved the first layer)
- within a given spread, the magnitude mask beat the random mask at global $5\%$ (15 or 16 vs. 9 of 30) but not at the other settings (27 vs. 28 at global $10\%$; 20 vs. 18 per layer)
- the paper's key claim, that the **original initial weights** matter, is **not reproduced in this one-shot setting**: original and new are tied (global 15 vs. 16 and 27 vs. 27; per layer 20 vs. 17)
- caveats
    - one pruning step, where the paper prunes iteratively; one small problem; a net far smaller than the paper's
    - the initialization (He, scaled for 50 inputs per unit) may be too small for units that keep only a few weights
    - 30 runs per bar: differences of a few runs are noise, and 6 to 7 runs (15 or 16 vs. 9) is borderline
- the mask comes from the trained net, so "a trainable sub-network exists" is built in; the test is whether it trains **from the start**
- **not tested**: the "bigger lottery" reading (wider nets contain more tickets); that needs the same test at several widths

## A practical rule, and does the paradox survive?

Weights in a dense net with 4 hidden layers of $w$ units (2 inputs, 1 output, no biases): $2w + 3w^2 + w$: $w=3$: $36$; $w=5$: $90$; $w=10$: $330$; $w=50$: $7650$.

- give each layer **enough units**: on this toy (2-D input, 2 classes, 4 layers) about 5 or more. With too few, ReLU layers often die (44% of 2-unit nets are dead at initialization, and some more die early in training) and tanh nets train slowly
- beyond that, extra width **did not** make training more reliable here (5, 10 and 50 units all succeed, 19/20 to 20/20), and 10% to 19% dead units are normal
- the warning signs: a **whole layer** dead, or training accuracy stuck at exactly chance: then change the seed, the initialization or the activation
- a plain dense net with 5 to 10 units (90 to 330 weights, counting weights only) trained in 20/20 runs
    - sparse nets found by pruning a 50-unit net: 382 weights worked in about half of the runs, 765 weights in $57\%$ to $93\%$ (different epochs and seeds: only a rough comparison)
    - in this toy, pruning did not give a small net that trains more easily than a plain dense one; it needs training again. The paper's claim is about matching the full net's accuracy, which we did not reach at these sparsities
- **the paradox**: it depends on what "what it needs" means
    - shrink to the **smallest net that could represent the task** (2 units: best runs reach $95\%$ to $97\%$) and it fails to train: **confirmed**
    - shrink to the **live units** of the big net (about $19\%$ of its units are dead, so roughly 40 of 50) and you land far above the threshold (not run, but 10 and 50 units both train; a 5-unit net has about $15\%$ dead, so about 4 live units, near the threshold, and width 4 was not run)
    - "many units contribute little" was measured only as dead units
    - beyond the threshold wide nets are **not** easier to train on this toy (we measured reliability, not speed)
- one problem, one set of seeds, one learning rate; on a real problem the threshold depends on the input, the depth and the data: start wide enough, and watch for dead layers and failing runs
- regularization to control overfitting is a separate topic (see [Learning-rate schedules and regularization](Training_Neural_Networks_Tweaks.ipynb)); back to the overview: [Small decisions, big effects](Training_Neural_Networks_Small_Decisions.ipynb)